# 直接读取你 WSL 已识别的全部逐日 GLORYS 流轴，绘制 119–147°E/20–42°N 六断面图。

完整代码内嵌，不依赖旁边的 `.py` 文件；根据情况修改最后一个 Cell 的 ROOT。

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Figure 2: daily identified GLORYS12V1 50 m Kuroshio axes (not monthly).
Runs on the user's WSL where immutable daily NPZ actually live. Requires numpy,
matplotlib, pandas, scipy; optional xarray / NetCDF4 only for land-mask background.
Example:
python R80_Figure2_Daily_GLORYS_Axis_SixTransects_WSL.py \
 --root /mnt/d/KuroshioDaily1993_2025 \
 --end 2024-12-31 --output /mnt/d/KuroshioDaily1993_2025/R80_review_figures
This script never modifies source axes and does not require re-extraction.
"""
from __future__ import annotations
import argparse,glob,re,warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from scipy.ndimage import gaussian_filter1d
SECTIONS={
 'A':((121.50,23.50),(125.00,22.75)),
 'B':((121.50,26.25),(124.25,24.25)),
 'PN':((126.00,29.00),(128.15,27.30)),
 'C':((130.25,31.25),(132.50,27.00)),
 'D':((137.00,34.75),(138.00,27.75)),
 'E':((141.75,40.00),(145.00,31.50)),
}
def simple_smooth_path(path, sigma_km=10, max_displace_km=12):
    """Display-only gentle Gaussian smoothing along individual path arclength.
    Never smooth across separate daily axes; endpoints preserved exactly.
    """
    xy=np.asarray(path,float)
    if len(xy)<10:return xy
    lat_mid=np.deg2rad((xy[:-1,1]+xy[1:,1])/2)
    d=np.sqrt((111.32*np.cos(lat_mid)*np.diff(xy[:,0]))**2+(111.32*np.diff(xy[:,1]))**2)
    xdist=np.r_[0,np.cumsum(d)]
    good=np.r_[True,d>.03]
    if good.sum()<10 or xdist[-1]<30:return xy
    s=np.arange(0,xdist[-1],5)
    if len(s)<10:return xy
    s=np.r_[s,xdist[-1]] if s[-1]!=xdist[-1] else s
    x=np.interp(s,xdist[good],xy[good,0]);y=np.interp(s,xdist[good],xy[good,1]);xyi=np.column_stack([x,y])
    smooth=gaussian_filter1d(xyi, sigma=max(.01,sigma_km/5),axis=0,mode='nearest')
    smooth[[0,-1]]=xyi[[0,-1]]
    dx=(smooth[:,0]-xyi[:,0])*111.32*np.cos(np.deg2rad(xyi[:,1]));dy=(smooth[:,1]-xyi[:,1])*111.32
    delta=np.sqrt(dx*dx+dy*dy)
    fac=np.minimum(1,max_displace_km/np.maximum(delta,.0001))
    res=xyi+(smooth-xyi)*fac[:,None]
    res[[0,-1]]=xyi[[0,-1]]
    return res

def load_npz(path):
    with np.load(path,allow_pickle=False) as z:
        a=np.asarray(z['lon'],float).reshape(-1)
        b=np.asarray(z['lat'],float).reshape(-1)
    x=np.column_stack([a,b]);x=x[np.isfinite(x).all(axis=1)]
    if len(x)<15 or not ((x[:,0]>=115)&(x[:,0]<=153)&(x[:,1]>=15)&(x[:,1]<=45)).all():
        raise ValueError('invalid axis coordinates')
    return x

def optional_land_shading(ax,root):
    # Optional real land mask from CMEMS ADT; no synthetic coastline ever drawn.
    fs=list(Path('/mnt/f/01-CMEMS/SEALEVEL_GLO_PHY_L4_MY_008_047').rglob('dt_global_allsat_phy_l4_202001*.nc'))[:1]
    if not fs:return False
    try:
        import xarray as xr
        with xr.open_dataset(fs[0],decode_times=False) as ds:
            data=ds['adt']; data=data.isel({d:0 for d in data.dims if d not in ['longitude','latitude']})
            roi=data.sel(longitude=slice(119,147),latitude=slice(20,42))
            lons=roi.longitude.values;lats=roi.latitude.values;v=roi.values
            if lons[0]>lons[-1]:lons=lons[::-1];v=v[:,::-1]
            if lats[0]>lats[-1]:lats=lats[::-1];v=v[::-1]
            land=(~np.isfinite(v)).astype(float)
            ax.contourf(lons,lats,land,levels=[.5,1.5],colors=['#eceff0'],zorder=0)
        return True
    except Exception as exc:
        warnings.warn('Could not show land mask from real ADT field: '+str(exc))
        return False

def main():
    p=argparse.ArgumentParser()
    p.add_argument('--root',default='/mnt/d/KuroshioDaily1993_2025')
    p.add_argument('--start',default='1993-01-01');p.add_argument('--end',default='2024-12-31')
    p.add_argument('--output',default='')
    p.add_argument('--daily-stride',type=int,default=1,help='Plot every daily path by default; use 5 for speed-focused preview')
    p.add_argument('--smooth-km',type=float,default=10)
    p.add_argument('--max-lines',type=int,default=0,help='0 means all date-matched daily paths')
    args=p.parse_args()
    root=Path(args.root);out=Path(args.output) if args.output else root/'R80_review_figures'
    figs=out/'figures';tables=out/'tables';figs.mkdir(parents=True,exist_ok=True);tables.mkdir(parents=True,exist_ok=True)
    folder=root/'final/tracks_final'
    if not folder.is_dir():raise FileNotFoundError(f'GLORYS daily axes not found: {folder}')
    fs=[]
    for f in folder.glob('????????.npz'):
        if f.stem.isdigit():
            day=pd.to_datetime(f.stem,format='%Y%m%d',errors='coerce')
            if pd.notna(day) and pd.Timestamp(args.start)<=day<=pd.Timestamp(args.end):fs.append((day,f))
    fs.sort(key=lambda v:v[0]);fs=fs[::max(1,args.daily_stride)]
    if args.max_lines>0:fs=fs[:args.max_lines]
    if not fs:raise FileNotFoundError(f'No actual daily GLORYS axes between {args.start} and {args.end}')
    paths=[];profile=[];date_records=[]
    # Mean path: identical normalized arclength coordinate ONLY FOR MAP DISPLAY.
    # Does not enter ADT-GLORYS D(t) statistics or numerical model evaluation.
    for i,(date,f) in enumerate(fs,1):
        try:
            x=load_npz(f)
            x=simple_smooth_path(x,args.smooth_km)
            paths.append(x)
            d=np.sqrt((111.32*np.cos(np.deg2rad((x[1:,1]+x[:-1,1])/2))*np.diff(x[:,0]))**2+
                      (111.32*np.diff(x[:,1]))**2)
            s=np.r_[0,np.cumsum(d)]
            if s[-1]>100 and np.all(np.diff(s)>0):
                q=np.linspace(0,s[-1],220)
                profile.append(np.column_stack([np.interp(q,s,x[:,0]),np.interp(q,s,x[:,1])]))
            date_records.append({'date':date.date().isoformat(),'npz':str(f),'n_points':len(x),'status':'ok'})
        except Exception as e:
            date_records.append({'date':date.date().isoformat(),'npz':str(f),'status':'fail:'+str(e)})
        if i%1500==0:print('Read daily GLORYS axes',i,'/',len(fs),flush=True)
    if not paths:raise RuntimeError('No valid daily GLORYS path')
    fig,ax=plt.subplots(figsize=(12.2,9.4));fig.subplots_adjust(left=.10,right=.965,bottom=.095,top=.93)
    have_land=optional_land_shading(ax,root)
    collection=LineCollection(paths,colors='#7e8790',linewidths=.29,alpha=.036,zorder=2,rasterized=True)
    ax.add_collection(collection)
    if profile:
        pr=np.stack(profile)
        mu=np.nanmean(pr,axis=0)
        mu=simple_smooth_path(mu,18,12)
        ax.plot(mu[:,0],mu[:,1],color='#249267',lw=2.8,zorder=4,label='Multi-year mean axis (display-only)')
    for i,(name,(p0,p1)) in enumerate(SECTIONS.items()):
        ax.plot([p0[0],p1[0]],[p0[1],p1[1]],lw=2.0,color='#bc4c51',zorder=5)
        frac=.67;where=np.array(p0)*(1-frac)+np.array(p1)*frac
        offsets={'A':(.25,-.5),'B':(-.6,.2),'PN':(.25,.45),'C':(.3,.45),'D':(.5,-.1),'E':(.45,-.5)}
        dx,dy=offsets[name]
        ax.text(where[0]+dx,where[1]+dy,name,color='#b23841',fontsize=16,weight='semibold',zorder=7)
    ax.set_xlim(119,147);ax.set_ylim(20,42)
    ax.set_xticks(np.arange(120,148,4));ax.set_yticks(np.arange(20,44,4))
    ax.set_xticklabels([f'{i}°E' for i in np.arange(120,148,4)],fontsize=13)
    ax.set_yticklabels([f'{i}°N' for i in np.arange(20,44,4)],fontsize=13)
    ax.set_aspect(1/np.cos(np.deg2rad(31)),adjustable='box')
    ax.grid(lw=.6,color='#aab3b8',alpha=.26)
    ax.set_xlabel('Longitude',fontsize=15);ax.set_ylabel('Latitude',fontsize=15)
    ax.set_title('Daily identified Kuroshio axes and six diagnostic transects',fontsize=17.5,pad=16,weight='semibold')
    # custom legend separate from grey opacity effect
    from matplotlib.lines import Line2D
    ax.legend(handles=[Line2D([0],[0],color='#7e8790',lw=2,label='Daily GLORYS 50 m axis'),
                       Line2D([0],[0],color='#249267',lw=2.8,label='Mean axis'),
                       Line2D([0],[0],color='#bc4c51',lw=2,label='Six transects')],
              frameon=False,fontsize=12,loc='lower right')
    stem='Figure2_daily_GLORYS_axis_6_transects'
    fig.savefig(figs/(stem+'.png'),dpi=300,facecolor='white');fig.savefig(figs/(stem+'.pdf'),facecolor='white');plt.close(fig)
    pd.DataFrame(date_records).to_csv(tables/'Figure2_daily_axis_read_audit.csv',index=False,encoding='utf-8-sig')
    summary={'input_paths':len(fs),'valid_paths':len(paths),'start_requested':args.start,'end_requested':args.end,'smooth_km_display_only':args.smooth_km,
             'daily_stride':args.daily_stride,'have_real_adt_land_mask':bool(have_land)}
    (tables/'Figure2_build_metadata.json').write_text(__import__('json').dumps(summary,ensure_ascii=False,indent=2),encoding='utf8')
    print('[DONE] Figure 2 plot:',figs/(stem+'.png'));print(summary)



In [ ]:
from pathlib import Path
import sys
ROOT = Path('/mnt/d/KuroshioDaily1993_2025')
OUT = ROOT / 'R80_review_figures'
sys.argv = ['script', '--root', str(ROOT), '--start', '1993-01-01', '--end', '2024-12-31', '--daily-stride', '1', '--output', str(OUT)]
main()